# wiki_cartoon — FLUX.2-dev LoRA inference (RunPod Pod)

RunPod-native version of the Colab notebook. Generate an Emslie clay-sculpt caricature with your
trained **`lora-cartoon`** FLUX.2-dev LoRA from a **prompt** (built from the caption lexicon) and,
optionally, a **reference photo** (FLUX.2's native image conditioning).

## Why this differs from the Colab notebook
On a 48GB+ pod there is **no need for 4-bit quant or CPU offload** — the model runs in full bf16,
which is what was causing the OOM cliff on Colab's 24GB L4. This notebook loads plain weights, saves
to the pod's persistent `/workspace`, and reads the HF token from an env var (no hard-coded secret).

## ⚠️ Read first — runtime requirements
FLUX.2-dev is huge (~32B transformer + Mistral-Small-3.1 text encoder).

| GPU | VRAM | This notebook (bf16, no offload) |
|---|---|---|
| A6000 / L40S | 48GB | **Yes** — comfortable, recommended sweet spot |
| A100 / H100 | 80GB | Yes — zero memory worry |
| L4 / 4090 | 24GB | **No** in bf16 — set `USE_4BIT = True` in the load cell (re-creates the Colab path) |

## Setup (do once, before running cells)
1. **Deploy a Pod** with a 48GB+ GPU and a **PyTorch** template. Attach a **Network Volume** mounted
   at `/workspace` so the multi-GB base weights are cached across pod restarts (the big win vs Colab).
2. **Set your HF token as an env var** on the pod (RunPod → Pod → Edit → Environment Variables):
   `HF_TOKEN=hf_...` from an account that has **accepted the FLUX.2-dev license** and can **read** the
   LoRA repo. (Or just run the auth cell and paste it once — but env var is cleaner and avoids committing it.)
3. Open Jupyter Lab on the pod and run the cells top to bottom.

## 1. Check the GPU you were given

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import psutil
print(f'System RAM: {psutil.virtual_memory().total/1e9:.0f} GB')
# Want 48GB+ for the bf16 path below. On 24GB, set USE_4BIT=True in the load cell.

## 2. Install dependencies
FLUX.2 support lives in recent diffusers — install from git to be safe. Cached on the network volume
if you `pip install` into it, otherwise re-runs on each fresh pod (fast after the first time).

In [ ]:
%pip install -q git+https://github.com/huggingface/diffusers.git
%pip install -q -U transformers accelerate bitsandbytes peft safetensors sentencepiece
print('Done. If diffusers was already imported this session, restart the kernel then continue from cell 3.')

## 3. Authenticate with Hugging Face
Reads `HF_TOKEN` from the pod environment (set it in the RunPod pod config). If it isn't set, paste it
into the prompt once — it is **not** stored in the notebook. The token must come from an account that has
accepted the FLUX.2-dev license and can read your `Guido/cartoon_lora` repo.

In [ ]:
import os, getpass
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF_TOKEN (input hidden): ').strip()
    os.environ['HF_TOKEN'] = HF_TOKEN

# Cache the multi-GB base weights on the persistent volume so pod restarts don't re-download.
os.environ.setdefault('HF_HOME', '/workspace/hf_cache')
login(token=HF_TOKEN)
print('Authenticated. HF cache at', os.environ['HF_HOME'])

## 4. Settings
The LoRA is pulled straight from your Hugging Face repo. Outputs save to the pod's persistent
`/workspace` — download them from the Jupyter file browser (right-click → Download).

In [ ]:
LORA_REPO = 'Guido/cartoon_lora'
LORA_WEIGHT_NAME = 'my_first_lora_v2.safetensors'
OUTPUT_DIR = '/workspace/outputs'
LORA_SCALE = 1.0
# Lower GUIDANCE (2.5–3) when using a reference photo; ~4 for prompt-only.
GUIDANCE = 3.5
STEPS = 28
WIDTH = 1024
HEIGHT = 1024
SEED = 42

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('LoRA:', f'{LORA_REPO}/{LORA_WEIGHT_NAME}')
print('Output:', OUTPUT_DIR)

## 5. Load FLUX.2-dev + your LoRA
First run downloads the gated base components (several GB) to `/workspace/hf_cache` — slow once, then
cached on the volume. On a 48GB+ GPU this loads in full bf16, no offload (the OOM-free path). Only set
`USE_4BIT = True` if you are stuck on a 24GB GPU.

In [ ]:
import torch
from diffusers import Flux2Pipeline, Flux2Transformer2DModel

REPO = 'black-forest-labs/FLUX.2-dev'
dtype = torch.bfloat16
USE_4BIT = False  # True only on a 24GB GPU (recreates the Colab quant+offload path)

if USE_4BIT:
    # Fallback for small GPUs: 4-bit transformer + CPU offload.
    transformer = Flux2Transformer2DModel.from_pretrained(
        'diffusers/FLUX.2-dev-bnb-4bit', subfolder='transformer',
        torch_dtype=dtype, device_map='cpu',
    )
    pipe = Flux2Pipeline.from_pretrained(REPO, transformer=transformer, torch_dtype=dtype)
    pipe.enable_model_cpu_offload()
else:
    # 48GB+ path: full bf16, whole pipeline on the GPU. No quant, no offload.
    pipe = Flux2Pipeline.from_pretrained(REPO, torch_dtype=dtype)
    pipe.to('cuda')

pipe.load_lora_weights(LORA_REPO, weight_name=LORA_WEIGHT_NAME, adapter_name='cartoon')
pipe.set_adapters('cartoon', LORA_SCALE)
print(f'Loaded LoRA {LORA_REPO}/{LORA_WEIGHT_NAME} at scale {LORA_SCALE}. Ready.')

## 6. (Optional) Reference photo
FLUX.2 conditions on the image natively. Upload one (or a few) into `/workspace` via the Jupyter file
browser, then list the path(s) below. Leave `REF_PATHS = []` for prompt-only generation.

**Caricature caveat:** the reference anchors *who the person is* but also resists the exaggeration.
Lean on the prompt for the distortion; if likeness dominates and the caricature is too tame, push the
deltas harder in the prompt or drop the reference — lowering guidance is not the lever.

In [ ]:
from PIL import Image

REF_PATHS = []  # e.g. ['/workspace/face.jpg']

ref_images = None
if REF_PATHS:
    ref_images = []
    for p in REF_PATHS:
        img = Image.open(p).convert('RGB')
        img.thumbnail((1024, 1024))
        ref_images.append(img)
    print(f'{len(ref_images)} reference image(s) loaded.')
    display(ref_images[0])
else:
    print('No reference — will generate from the prompt alone.')

## 7. Prompt → generate → save
Write the prompt in **caption-lexicon** phrasing, led by `lora-cartoon` (what the LoRA was trained on).

In [ ]:
PROMPT = 'lora-cartoon, a 3D sculpted character caricature bust, smooth grey clay ZBrush render, neutral studio lighting. Three-quarter view. An exaggerated caricature of a bald older man with a big domed crown, big ears, and a wide toothy grin. Against a dark grey gradient background.'

import datetime

gen = torch.Generator(device='cuda').manual_seed(int(SEED))
kwargs = dict(prompt=PROMPT, num_inference_steps=int(STEPS),
              guidance_scale=float(GUIDANCE), width=int(WIDTH), height=int(HEIGHT),
              generator=gen)
if ref_images:
    kwargs['image'] = ref_images  # FLUX.2 accepts a list of reference images

image = pipe(**kwargs).images[0]

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
out_path = f'{OUTPUT_DIR}/cartoon-{stamp}-seed{SEED}.png'
image.save(out_path)
print('Saved:', out_path)
display(image)

### Iterate
Re-run cell 7 with a new prompt / seed. To change LoRA strength, edit `LORA_SCALE` in cell 4 and
re-run the last two lines of cell 5 (`set_adapters`). For a different reference, edit `REF_PATHS` in
cell 6. Download results from the Jupyter file browser at `/workspace/outputs`.

**Stop the pod when done** — RunPod bills per second while it runs. A network volume keeps your weights
cached for next time.